> **Chapter Map:** This notebook is the companion for **Chapter 1** (Bridge Module).

# NB1: Bridge Module — Python Computational Tools for Kinetic Data Analysis

## Learning Objectives

After completing this notebook, you will be able to:

1. Load and inspect experimental kinetic data from CSV files using pandas
2. Create publication-quality plots with proper axis labels, units, and formatting
3. Perform linear regression on transformed kinetic data (Arrhenius analysis)
4. Extract kinetic parameters (activation energy, pre-exponential factor) with statistical confidence
5. Conduct residual analysis to evaluate the quality of a fit
6. Wrap analysis workflows into reusable Python functions

## Background

This notebook introduces the core Python computational workflow that you will use throughout this course. The workflow consists of four main steps:

1. **Data Input**: Reading experimental data from files
2. **Visualization**: Creating informative plots
3. **Analysis**: Fitting models to data and extracting parameters
4. **Validation**: Checking the quality of fits using residual analysis

We will demonstrate this workflow using **Arrhenius analysis** - a fundamental technique for determining activation energy from temperature-dependent rate constant data.

### The Arrhenius Equation

The temperature dependence of reaction rate constants follows the Arrhenius equation:

$$k = A \exp\left(-\frac{E_a}{RT}\right)$$

Taking the natural logarithm of both sides:

$$\ln k = \ln A - \frac{E_a}{R} \cdot \frac{1}{T}$$

This is a linear equation: $y = b + m \cdot x$ where:
- $y = \ln k$
- $x = 1/T$
- Slope $m = -E_a/R$
- Intercept $b = \ln A$

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Set default plot style for publication-quality figures
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constant
R = 8.314  # Gas constant in J/(mol*K)

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

print("Setup complete. Libraries imported successfully.")

---
## Part 1: Reading CSV Data

In experimental work, data is typically stored in spreadsheet formats. We will use pandas to read and manipulate tabular data. First, let's create a sample dataset based on the Arrhenius worked example from the lecture notes.

### 1.1 Creating and Saving Sample Data

We have experimental rate constants measured at five different temperatures for a catalytic reaction.

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Try changing these values to explore different
# Arrhenius behaviors (e.g., higher Ea, fewer points).

# Experimental data from the worked example
# Reaction: Catalytic decomposition on metal oxide catalyst
temperature_K = np.array([450, 475, 500, 525, 550])  # Temperature in Kelvin
rate_constant = np.array([0.0234, 0.0891, 0.298, 0.876, 2.34])  # k in mol/(kg*s)

# =============================================

# Create a pandas DataFrame
data = pd.DataFrame({
    'Temperature_K': temperature_K,
    'k_mol_per_kg_s': rate_constant
})

# Display the data
print("Experimental Rate Constant Data")
print("=" * 40)
print(data.to_string(index=False))

In [ ]:
# Save to CSV file
csv_filename = 'arrhenius_data.csv'
data.to_csv(csv_filename, index=False)
print(f"Data saved to: {csv_filename}")

### 1.2 Reading Data from CSV

Now let's read the data back from the CSV file, as you would with real experimental data.

In [ ]:
# Read data from CSV file
df = pd.read_csv(csv_filename)

# Display basic information about the dataset
print("DataFrame Info:")
print("-" * 40)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print()
print("Data Preview:")
print("-" * 40)
df

In [ ]:
# Extract columns as numpy arrays for calculations
T = df['Temperature_K'].values  # Temperature array
k = df['k_mol_per_kg_s'].values  # Rate constant array

# Basic statistics
print("Summary Statistics:")
print("-" * 40)
print(df.describe())

---
## Part 2: Basic Plotting

Visualization is essential for understanding data before analysis. Let's create both a simple plot and a properly formatted publication-quality figure.

### 2.1 A Minimal Plot (What NOT to Submit)

This plot lacks essential elements for scientific communication.

In [ ]:
# Minimal plot - missing important elements
plt.figure(figsize=(6, 4))
plt.plot(T, k)
plt.show()

**Problems with this plot:**
- No axis labels
- No units
- No title or context
- No data point markers (only line)
- No grid for reading values

### 2.2 A Properly Formatted Plot

Scientific figures should be self-explanatory with clear labels, units, and appropriate formatting.

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Try changing marker style, colors, or figure size
# to explore matplotlib formatting options.
marker_style = 'o'          # Try 's', '^', 'D', 'v'
marker_size = 10            # Marker diameter in points
line_style = '-'            # Try '--', '-.', ':'
fig_width, fig_height = 8, 6
# =============================================

# Publication-quality plot
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

# Plot data with markers and line
ax.plot(T, k, 
        marker=marker_style,
        markersize=marker_size,
        markerfacecolor=COLORS['blue'],
        markeredgecolor='black',
        markeredgewidth=1.5,
        linestyle=line_style,
        linewidth=2,
        color=COLORS['blue'],
        label='Experimental data')

# Axis labels with units
ax.set_xlabel('Temperature, $T$ (K)')
ax.set_ylabel('Rate Constant, $k$ (mol kg$^{-1}$ s$^{-1}$)')

# Title describing the experiment
ax.set_title('Temperature Dependence of Reaction Rate Constant')

# Legend
ax.legend(loc='upper left')

# Grid for easier reading
ax.grid(True, alpha=0.3)

# Ensure all labels are visible
plt.tight_layout()
plt.show()

**Key elements of a good scientific plot:**
1. Clear axis labels with units
2. Descriptive title
3. Data markers visible (not just a line)
4. Legend identifying data series
5. Grid for reading approximate values
6. Colorblind-safe colors
7. Appropriate font sizes

---
## Part 3: Arrhenius Analysis

To extract the activation energy $E_a$ and pre-exponential factor $A$, we linearize the Arrhenius equation by plotting $\ln k$ versus $1/T$.

### 3.1 Data Transformation

In [ ]:
# Transform data for Arrhenius plot
inv_T = 1 / T  # 1/T in K^-1
ln_k = np.log(k)  # Natural logarithm of k

# Add transformed data to DataFrame for reference
df['1/T_K^-1'] = inv_T
df['ln_k'] = ln_k

print("Transformed Data for Arrhenius Analysis:")
print("-" * 60)
print(df.to_string(index=False))

### 3.2 Arrhenius Plot

In [ ]:
# Create Arrhenius plot
fig, ax = plt.subplots(figsize=(8, 6))

# Plot transformed data
ax.plot(inv_T * 1000, ln_k,  # Scale 1/T by 1000 for readability
        marker='s',
        markersize=10,
        markerfacecolor=COLORS['orange'],
        markeredgecolor='black',
        markeredgewidth=1.5,
        linestyle='none',  # Only markers, no connecting line
        label='Experimental data')

# Axis labels (note: 1/T scaled by 1000)
ax.set_xlabel('1000/$T$ (K$^{-1}$)')
ax.set_ylabel('ln($k$ / mol kg$^{-1}$ s$^{-1}$)')
ax.set_title('Arrhenius Plot: ln($k$) vs. 1/$T$')

ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: If the Arrhenius equation applies, these points should fall on a straight line.")

### 3.3 Linear Regression

We use `scipy.stats.linregress` to perform linear regression and extract the kinetic parameters.

In [ ]:
# Perform linear regression: ln(k) = ln(A) - (Ea/R) * (1/T)
# y = b + m*x where m = slope, b = intercept
result = stats.linregress(inv_T, ln_k)

# Extract regression results
slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2
std_err_slope = result.stderr
std_err_intercept = result.intercept_stderr

print("Linear Regression Results:")
print("=" * 50)
print(f"Slope     = {slope:.2f} +/- {std_err_slope:.2f} K")
print(f"Intercept = {intercept:.3f} +/- {std_err_intercept:.3f}")
print(f"R-squared = {r_squared:.6f}")

In [ ]:
# Extract Arrhenius parameters
# From slope = -Ea/R, so Ea = -slope * R
Ea = -slope * R  # Activation energy in J/mol
Ea_kJ = Ea / 1000  # Convert to kJ/mol

# Uncertainty in Ea
Ea_uncertainty = std_err_slope * R / 1000  # kJ/mol

# From intercept = ln(A), so A = exp(intercept)
A = np.exp(intercept)  # Pre-exponential factor in mol/(kg*s)

# Uncertainty in A (propagated from intercept uncertainty)
# d(A)/d(intercept) = A, so delta_A = A * delta_intercept
A_uncertainty = A * std_err_intercept

print("\nExtracted Arrhenius Parameters:")
print("=" * 50)
print(f"Activation Energy, Ea = {Ea_kJ:.1f} +/- {Ea_uncertainty:.1f} kJ/mol")
print(f"Pre-exponential Factor, A = {A:.2e} +/- {A_uncertainty:.2e} mol/(kg*s)")
print(f"\nQuality of fit: R-squared = {r_squared:.4f}")

### 3.4 Arrhenius Plot with Fitted Line

In [ ]:
# Create Arrhenius plot with fitted line
fig, ax = plt.subplots(figsize=(8, 6))

# Generate fitted line
inv_T_fit = np.linspace(inv_T.min() * 0.98, inv_T.max() * 1.02, 100)
ln_k_fit = intercept + slope * inv_T_fit

# Plot fitted line first (so it appears behind data points)
ax.plot(inv_T_fit * 1000, ln_k_fit,
        linestyle='-',
        linewidth=2,
        color=COLORS['vermillion'],
        label=f'Linear fit ($R^2$ = {r_squared:.4f})')

# Plot experimental data
ax.plot(inv_T * 1000, ln_k,
        marker='s',
        markersize=10,
        markerfacecolor=COLORS['blue'],
        markeredgecolor='black',
        markeredgewidth=1.5,
        linestyle='none',
        label='Experimental data')

# Add annotation with extracted parameters
textstr = (f'$E_a$ = {Ea_kJ:.1f} kJ/mol\n'
           f'$A$ = {A:.2e} mol/(kg s)')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=props)

# Axis labels
ax.set_xlabel('1000/$T$ (K$^{-1}$)')
ax.set_ylabel('ln($k$ / mol kg$^{-1}$ s$^{-1}$)')
ax.set_title('Arrhenius Analysis: Determination of $E_a$ and $A$')

ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 4: Residual Analysis

A high $R^2$ value does not guarantee that the model is appropriate. Residual analysis helps identify systematic deviations from the linear model.

### 4.1 Calculating Residuals

Residuals are the differences between observed and predicted values:
$$\text{residual}_i = y_{\text{observed},i} - y_{\text{predicted},i}$$

In [ ]:
# Calculate predicted values from the fitted line
ln_k_predicted = intercept + slope * inv_T

# Calculate residuals
residuals = ln_k - ln_k_predicted

# Display residuals
residual_df = pd.DataFrame({
    'T (K)': T,
    'ln(k) observed': ln_k,
    'ln(k) predicted': ln_k_predicted,
    'Residual': residuals
})

print("Residual Analysis:")
print("-" * 70)
print(residual_df.to_string(index=False, float_format='{:.4f}'.format))
print(f"\nSum of residuals: {residuals.sum():.6f} (should be ~0)")
print(f"Standard deviation of residuals: {residuals.std():.4f}")

### 4.2 Residual Plot

In [ ]:
# Create residual plot
fig, ax = plt.subplots(figsize=(8, 5))

# Plot residuals
ax.plot(inv_T * 1000, residuals,
        marker='o',
        markersize=10,
        markerfacecolor=COLORS['green'],
        markeredgecolor='black',
        markeredgewidth=1.5,
        linestyle='none',
        label='Residuals')

# Add horizontal line at zero
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Add +/- 2*std boundaries (approximate 95% confidence)
std_resid = residuals.std()
ax.axhline(y=2*std_resid, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=-2*std_resid, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.fill_between(ax.get_xlim(), -2*std_resid, 2*std_resid, 
                alpha=0.1, color='gray', label=r'$\pm 2\sigma$ band')

# Labels
ax.set_xlabel('1000/$T$ (K$^{-1}$)')
ax.set_ylabel('Residual (ln($k$) observed - predicted)')
ax.set_title('Residual Plot: Checking for Systematic Deviations')

ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Interpreting Residual Plots

**Good fit indicators:**
- Residuals are randomly scattered around zero
- No obvious pattern or trend
- Most points within $\pm 2\sigma$ band

**Warning signs (poor fit):**
- **Curved pattern**: Model is too simple (nonlinear relationship)
- **Trend (increasing/decreasing)**: Missing variable or wrong functional form
- **Clusters**: Data from different populations/conditions
- **Outliers**: Measurement errors or unusual conditions

In our case, the residuals show random scatter with no systematic pattern, confirming that the Arrhenius model is appropriate for this data.

---
## Part 5: Complete Workflow as a Reusable Function

To make our analysis reusable, we wrap the entire Arrhenius analysis into a function.

In [ ]:
def arrhenius_analysis(T, k, plot=True, verbose=True):
    """
    Perform Arrhenius analysis on rate constant vs temperature data.
    
    Fits the linearized Arrhenius equation: ln(k) = ln(A) - Ea/(R*T)
    to extract activation energy and pre-exponential factor.
    
    Parameters
    ----------
    T : array-like
        Temperature values in Kelvin
    k : array-like
        Rate constant values (any consistent units)
    plot : bool, optional
        If True, generate Arrhenius and residual plots (default: True)
    verbose : bool, optional
        If True, print results to console (default: True)
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'Ea_kJ_mol': Activation energy in kJ/mol
        - 'Ea_uncertainty': Uncertainty in Ea (kJ/mol)
        - 'A': Pre-exponential factor (same units as k)
        - 'A_uncertainty': Uncertainty in A
        - 'R_squared': Coefficient of determination
        - 'residuals': Array of residuals
    
    Examples
    --------
    >>> T = np.array([450, 475, 500, 525, 550])
    >>> k = np.array([0.0234, 0.0891, 0.298, 0.876, 2.34])
    >>> results = arrhenius_analysis(T, k)
    >>> print(f"Ea = {results['Ea_kJ_mol']:.1f} kJ/mol")
    """
    # Convert to numpy arrays
    T = np.asarray(T)
    k = np.asarray(k)
    
    # Gas constant
    R = 8.314  # J/(mol*K)
    
    # Transform data
    inv_T = 1 / T
    ln_k = np.log(k)
    
    # Linear regression
    result = stats.linregress(inv_T, ln_k)
    slope = result.slope
    intercept = result.intercept
    r_squared = result.rvalue**2
    std_err_slope = result.stderr
    std_err_intercept = result.intercept_stderr
    
    # Extract parameters
    Ea = -slope * R  # J/mol
    Ea_kJ = Ea / 1000  # kJ/mol
    Ea_uncertainty = std_err_slope * R / 1000  # kJ/mol
    
    A = np.exp(intercept)
    A_uncertainty = A * std_err_intercept
    
    # Calculate residuals
    ln_k_predicted = intercept + slope * inv_T
    residuals = ln_k - ln_k_predicted
    
    # Print results
    if verbose:
        print("\nArrhenius Analysis Results")
        print("=" * 50)
        print(f"Activation Energy, Ea = {Ea_kJ:.1f} +/- {Ea_uncertainty:.1f} kJ/mol")
        print(f"Pre-exponential Factor, A = {A:.2e} +/- {A_uncertainty:.2e}")
        print(f"R-squared = {r_squared:.6f}")
    
    # Generate plots
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Arrhenius plot
        ax1 = axes[0]
        inv_T_fit = np.linspace(inv_T.min() * 0.98, inv_T.max() * 1.02, 100)
        ln_k_fit = intercept + slope * inv_T_fit
        
        ax1.plot(inv_T_fit * 1000, ln_k_fit, '-', color='#D55E00', linewidth=2,
                 label=f'Linear fit ($R^2$ = {r_squared:.4f})')
        ax1.plot(inv_T * 1000, ln_k, 's', markersize=10, 
                 markerfacecolor='#0072B2', markeredgecolor='black',
                 markeredgewidth=1.5, linestyle='none', label='Data')
        
        textstr = f'$E_a$ = {Ea_kJ:.1f} kJ/mol\n$A$ = {A:.2e}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax1.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=11,
                 verticalalignment='top', bbox=props)
        
        ax1.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax1.set_ylabel('ln($k$)')
        ax1.set_title('Arrhenius Plot')
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        
        # Residual plot
        ax2 = axes[1]
        ax2.plot(inv_T * 1000, residuals, 'o', markersize=10,
                 markerfacecolor='#009E73', markeredgecolor='black',
                 markeredgewidth=1.5)
        ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
        std_resid = residuals.std()
        ax2.axhline(y=2*std_resid, color='gray', linestyle='--', alpha=0.7)
        ax2.axhline(y=-2*std_resid, color='gray', linestyle='--', alpha=0.7)
        
        ax2.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax2.set_ylabel('Residual')
        ax2.set_title('Residual Plot')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Return results as dictionary
    return {
        'Ea_kJ_mol': Ea_kJ,
        'Ea_uncertainty': Ea_uncertainty,
        'A': A,
        'A_uncertainty': A_uncertainty,
        'R_squared': r_squared,
        'residuals': residuals
    }

### 5.1 Using the Function

In [ ]:
# Demonstrate the reusable function with our data
results = arrhenius_analysis(T, k)

In [ ]:
# Access individual results from the returned dictionary
print("Accessing individual results:")
print("-" * 40)
print(f"Ea = {results['Ea_kJ_mol']:.2f} kJ/mol")
print(f"A = {results['A']:.3e}")
print(f"R-squared = {results['R_squared']:.6f}")

---
## Summary

In this notebook, we have covered the complete data analysis workflow:

1. **Data I/O**: Created CSV files and loaded them with pandas
2. **Visualization**: Compared minimal vs. publication-quality plots
3. **Linear Regression**: Used `scipy.stats.linregress` for fitting
4. **Parameter Extraction**: Obtained $E_a$ and $A$ with uncertainties
5. **Residual Analysis**: Validated the fit quality
6. **Function Design**: Wrapped the workflow in a reusable function

### Key Takeaways

- Always include **units** and **axis labels** in plots
- Report **uncertainties** with extracted parameters
- Use **residual plots** to check for systematic deviations
- **$R^2$ alone is not sufficient** to validate a model
- Wrap analysis into **functions** for reproducibility

### Results from This Analysis

For the catalytic reaction studied:
- Activation energy: $E_a = 100$ kJ/mol (approximately)
- Pre-exponential factor: $A \approx 10^8$ mol/(kg s)
- Excellent linear fit ($R^2 > 0.999$) confirms Arrhenius behavior

**Next notebook:** NB2 covers adsorption, kinetics, and temperature dependence (Chapter 2).

---
## Exercises

### Exercise 1: Loading External Data

The file `arrhenius_data.csv` was created earlier. Practice loading it and verifying the data matches what we expect.

**Task:** Load the CSV file, print the first 3 rows, and calculate the mean temperature.

In [ ]:
# YOUR CODE HERE
# 1. Load the CSV file using pd.read_csv()
# 2. Display the first 3 rows using .head(3)
# 3. Calculate and print the mean temperature

pass  # Replace with your implementation

### Exercise 2: Plot Customization

Create a plot of k vs T using a **logarithmic y-axis** (since k spans two orders of magnitude).

**Task:** Use `ax.set_yscale('log')` to create a semi-log plot.

In [ ]:
# YOUR CODE HERE
# Create a plot with:
# - Linear x-axis (Temperature)
# - Logarithmic y-axis (Rate constant)
# - Proper labels with units
# - Data markers

pass  # Replace with your implementation

### Exercise 3: New Dataset Analysis

A second catalyst was tested with the following data:

| T (K) | k (mol/kg/s) |
|-------|-------------|
| 400   | 0.0052      |
| 425   | 0.018       |
| 450   | 0.056       |
| 475   | 0.16        |
| 500   | 0.42        |

**Task:** Use the `arrhenius_analysis` function to analyze this data and compare $E_a$ with the original catalyst.

In [ ]:
# YOUR CODE HERE
# 1. Create arrays for the new T and k data
# 2. Call arrhenius_analysis(T_new, k_new)
# 3. Compare Ea values between the two catalysts

pass  # Replace with your implementation

### Reflection Questions

1. Why do we plot ln(k) vs 1/T instead of k vs T for Arrhenius analysis?

2. If the residual plot showed a curved pattern instead of random scatter, what would that indicate about the Arrhenius model?

3. The pre-exponential factor A has the same units as k. What is the physical interpretation of A?

4. How would you modify the `arrhenius_analysis` function to also report the 95% confidence interval for $E_a$?